In [1]:
import dspy
from typing import Literal, List
dspy.configure_cache(
    enable_disk_cache=False,
    enable_memory_cache=False,
)


class CLassifyDomain(dspy.Signature):
        """
        Classify the study design described in the abstract.

        Available study-design labels:
            "randomized_controlled_trial", "nonrandomized_controlled_trial",
            "prospective_cohort", "retrospective_cohort", "case_control",
            "cross_sectional", "case_series", "case_report",
            "diagnostic_accuracy_study", "systematic_review_or_meta_analysis",
            "guideline_or_editorial_or_commentary",
            "methods_or_assay_development", "in_vitro", "animal_model",
            "imaging_only", "conference_abstract_or_poster",
            "other_or_unclear"

        **Primary design selection rules:**
        - Randomized allocation → randomized_controlled_trial
        - Nonrandomized comparator groups → nonrandomized_controlled_trial
        - Prospective follow-up of a group → prospective_cohort
        - Retrospective chart/registry review → retrospective_cohort
        - Explicit “cases vs controls” comparison → case_control
        - Single time-point measurement/survey/prevalence → cross_sectional
        - ≥2 patients without a control group → case_series
        - Single patient → case_report
        - Test/validation of diagnostic performance → diagnostic_accuracy_study
        - Systematic review or meta-analysis → systematic_review_or_meta_analysis
        - No primary data (guidelines, commentary, editorial) → guideline_or_editorial_or_commentary
        - Methods/assay development without clinical outcomes → methods_or_assay_development
        - In vitro experiments → in_vitro
        - Animal experiments → animal_model
        - Imaging-only analyses without clinical outcomes → imaging_only
        - Conference abstracts/posters → conference_abstract_or_poster
        - Anything unclear or mixed → other_or_unclear

        **Secondary design rules:**
        - secondary_designs must be a JSON array.
        - Choose 0–3 additional labels if they meaningfully apply.
        - Use only labels from the primary-design list.
        - Use [] if none apply.

        **Task:**
        Read the abstract and output:
            (1) the single best-fitting primary_design
            (2) an optional list (0–3 items) of secondary_designs
        """
    
        abstract: str = dspy.InputField(
            desc="The Abstract text to classify into themes"
        )
        primary_design: Literal[
            "randomized_controlled_trial",
            "nonrandomized_controlled_trial",
            "prospective_cohort",
            "retrospective_cohort",
            "case_control",
            "cross_sectional",
            "case_series",
            "case_report",
            "diagnostic_accuracy_study",
            "systematic_review_or_meta_analysis",
            "guideline_or_editorial_or_commentary",
            "methods_or_assay_development",
            "in_vitro",
            "animal_model",
            "imaging_only",
            "conference_abstract_or_poster",
            "other_or_unclear",
        ] = dspy.OutputField(desc="The primary design classification")
        secondary_designs: List[
            Literal[
                      "randomized_controlled_trial",
            "nonrandomized_controlled_trial",
            "prospective_cohort",
            "retrospective_cohort",
            "case_control",
            "cross_sectional",
            "case_series",
            "case_report",
            "diagnostic_accuracy_study",
            "systematic_review_or_meta_analysis",
            "guideline_or_editorial_or_commentary",
            "methods_or_assay_development",
            "in_vitro",
            "animal_model",
            "imaging_only",
            "conference_abstract_or_poster",
            "other_or_unclear",
            ]
        ] = dspy.OutputField(desc="list of 0-3 secondary design classifications")


        

In [2]:
def create_DSPy_example(data):
    gold_standard = []
    for row in data:
        gold_standard.append(
            dspy.Example(
                # context = CONTEXT,
                abstract=row['abstract'],
                primary_design=row['primary_design'],
                secondary_designs=row['secondary_designs'],
            ).with_inputs("abstract"),
        )
    return gold_standard

In [3]:
import json
with open("../../validation/domain_classification/classification_results_ensembled_domain_classification_train.json", "r") as f:
    train = json.load(f)
with open("../../validation/domain_classification/classification_results_ensembled_domain_classification_val.json", "r") as f:
    val = json.load(f)
with open("../../validation/domain_classification/classification_results_ensembled_domain_classification_test.json", "r") as f:
    test = json.load(f)
gold_standard_train = create_DSPy_example(train)
gold_standard_val = create_DSPy_example(val)
gold_standard_test = create_DSPy_example(test)

In [4]:
def gepa_feedback_metric(gold: dspy.Example,
                         pred: dspy.Prediction,
                         trace=None,
                         pred_name=None,
                         pred_trace=None):
    # simple exact-match score on the decision
    pred_primary_design = getattr(pred, "primary_design", None)
    gold_primary_design = getattr(gold, "primary_design", None)
    
    score = 0
    if pred_primary_design == gold_primary_design:
        score = 1.0
    else:
        score = 0.0
    
    # brief feedback that GEPA can reflect on
    if score == 1.0:
        fb = "Decision matches gold. Keep citing PICOS elements clearly."
    else:
        # Assuming your student output has 'reasoning', 'classification', and 'confidence'
# and your gold data has 'classification' and 'abstract' (as context)

        fb = (
            f"Decision does not match gold. "
            f"Predicted: {pred_primary_design}. "
            f"Gold: {gold_primary_design}. "
            f"Review the abstract and ensure correct classification."
        )

    # Return only the score for GEPA
    return dspy.Prediction(score=score, feedback=fb)

In [5]:
import json
import os

student_llm_string = "openai/gpt-5-mini"
screener_results_path = "../../classifier/domain_classification/classification_results_gpt_groundtruth.json"
results_path = "../../results/domain_classification/classification_results_gpt_original.json"
results_path_save = "../../results/domain_classification/classification_results_gpt_original.jsonl"

API_KEY = os.getenv("openrouter_api_key")
student_lm = dspy.LM(
    model=student_llm_string,       # e.g. "openrouter/google/gemini-2.0-flash-001"
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    # (plus any model_params like temperature, max_tokens, etc)
    # temperature=1.0, top_p=1.0, seed=42
    temperature=1.0, max_tokens = 350000,
)
dspy.configure(lm=student_lm)
reflector_lmm_string = "openai/gpt-4o-mini"
reflector_lm = dspy.LM(
    model=reflector_lmm_string,       # e.g. "openrouter/google/gemini-2.0-flash-001"
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    # temperature=1.0, top_p=0.95, seed=123
    temperature=1.0, max_tokens = 100000,
    )

In [6]:
gepa = dspy.GEPA(
    metric=gepa_feedback_metric,      # your feedback metric
    reflection_lm=reflector_lm,  
    # only the reflector is stochastic
    auto = 'medium',
    reflection_minibatch_size=5,
    use_merge=True,
    max_merge_invocations=10,
    track_stats=True,
    # skip_perfect_score=False,
)

compiled_screener = gepa.compile(
    student=dspy.ChainOfThought(CLassifyDomain),
    trainset=gold_standard_train, # minimal viable setup
    valset=gold_standard_val,
)
cost = sum([x['cost'] for x in student_lm.history if x['cost'] is not None])  # cost in USD, as calculated by LiteLLM for certain providers
print(cost)

# --- save and load as before ---
compiled_screener.save(path=screener_results_path)
# same lm used for a minimal setup in this example

2025/11/25 10:46:07 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 825 metric calls of the program. This amounts to 7.86 full evals on the train+val set.
2025/11/25 10:46:07 INFO dspy.teleprompt.gepa.gepa: Using 27 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget.
GEPA Optimization:   0%|          | 0/825 [00:00<?, ?rollouts/s]2025/11/25 10:46:35 INFO dspy.evaluate.evaluate: Average Metric: 27.0 / 27 (100.0%)
2025/11/25 10:46:35 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 1.0
GEPA Optimization:   3%|▎         | 27/825 [00:28<13:54,  1.05s/rollouts]2025/11/25 10:46:35 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 1.0


Average Metric: 5.00 / 5 (100.0%): 100%|██████████| 5/5 [00:07<00:00,  1.50s/it]

2025/11/25 10:46:43 INFO dspy.evaluate.evaluate: Average Metric: 5.0 / 5 (100.0%)
2025/11/25 10:46:43 INFO dspy.teleprompt.gepa.gepa: Iteration 1: All subsample scores perfect. Skipping.
2025/11/25 10:46:43 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Reflective mutation did not propose a new candidate
GEPA Optimization:   4%|▍         | 32/825 [00:35<15:07,  1.14s/rollouts]2025/11/25 10:46:43 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Selected program 0 score: 1.0



Average Metric: 5.00 / 5 (100.0%): 100%|██████████| 5/5 [00:08<00:00,  1.79s/it]

2025/11/25 10:46:52 INFO dspy.evaluate.evaluate: Average Metric: 5.0 / 5 (100.0%)
2025/11/25 10:46:52 INFO dspy.teleprompt.gepa.gepa: Iteration 2: All subsample scores perfect. Skipping.
2025/11/25 10:46:52 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Reflective mutation did not propose a new candidate
GEPA Optimization:   4%|▍         | 37/825 [00:44<17:00,  1.30s/rollouts]2025/11/25 10:46:52 INFO dspy.teleprompt.gepa.gepa: Iteration 3: Selected program 0 score: 1.0


...

2025/11/25 11:17:20 INFO dspy.evaluate.evaluate: Average Metric: 5.0 / 5 (100.0%)
2025/11/25 11:17:20 INFO dspy.teleprompt.gepa.gepa: Iteration 77: All subsample scores perfect. Skipping.
2025/11/25 11:17:20 INFO dspy.teleprompt.gepa.gepa: Iteration 77: Reflective mutation did not propose a new candidate
GEPA Optimization: 100%|█████████▉| 821/825 [31:12<00:08,  2.05s/rollouts]2025/11/25 11:17:20 INFO dspy.teleprompt.gepa.gepa: Iteration 78: Selected program 12 score: 1.0



Average Metric: 4.00 / 5 (80.0%): 100%|██████████| 5/5 [00:07<00:00,  1.52s/it] 

2025/11/25 11:17:28 INFO dspy.evaluate.evaluate: Average Metric: 4.0 / 5 (80.0%)


2025/11/25 11:17:48 INFO dspy.teleprompt.gepa.gepa: Iteration 78: Proposed new text for predict: Classify Medical Research Study Designs from Abstracts

**Task Overview:**
Your goal is to classify the design of medical research studies based on provided abstracts. Each abstract summarizes a study and includes elements such as objectives, methodologies, participant information, results, and other relevant details crucial for accurate classification. You will analyze each abstract and classify it into specific predefined categories of study design.

**Input Format:**
- Each input consists of a single abstract summarizing a medical research study.

**Study-Design Labels:**
- "randomized_controlled_trial"
- "nonrandomized_controlled_trial"
- "prospective_cohort"
- "retrospective_cohort"
- "case_control"
- "cross_sectional"
- "case_series"
- "case_report"
- "diagnostic_accuracy_study"
- "systematic_review_or_meta_analysis"
- "guideline_or_editorial_or_commentary"
- "methods_or_assay_develop

1.3819622499999995


In [7]:
domain_classifier = dspy.ChainOfThought(CLassifyDomain)
domain_classifier.load(path=screener_results_path)
test_results = []
total_accuracy = 0.0
for example in gold_standard_test:
    pred = domain_classifier(abstract=example.abstract)
    if pred.primary_design == example.primary_design:
        accuracy = 1.0
    else:
        accuracy = 0.0
    test_results.append({
        "abstract": example.abstract,
        "primary_design": pred.primary_design,
        "secondary_designs": pred.secondary_designs,
        "accuracy": accuracy,
    })
    total_accuracy += accuracy
print(f"Test Accuracy: {total_accuracy / len(gold_standard_test):.2f}")

Test Accuracy: 0.93
